# 04_Testing_and_deploying.ipynb

This is the fourth and last notebook of the project, where the final models will be tested under noise, feature dropping and through interpretability algorithms

After having chosen a main and backup model for production, a demo on Hugging Face spaces will be hosted to simulate real-world production environment

## Key Tests and Operational Workflow

### 1. Monte Carlo Noise Injection
To bridge the gap between abstract statistical validation and practical corporate risk management, we reject static performance metrics. Instead, we implement a 5-fold Cross-Validation framework coupled with a stochastic Monte Carlo simulator that injects progressive noise across continuous, discrete, and categorical features controlled by an intensity parameter $\sigma \in [0, 0.50]$.

To identify the optimal deployment strategy, we evaluate the financial decay curve using a **Stochastic Expected Profit ($EP$) Framework**. Rather than judging models on isolated noise levels, we compute a weighted average of financial returns across the entire noise spectrum:
$$EP = \sum_{i} X_i \times \text{Profit}(\sigma_i)$$

We define two distinct coefficient vectors ($X_i$) representing parallel corporate operating environments to select our first two specialized architectures:
* **The "Nominal/Stable" World Weighting:** Simulates standard operational conditions where infrastructure and data collection pipelines are stable. High probability weights are concentrated at $\sigma = 0.00$ and $\sigma = 0.05$. The estimator that maximizes this metric will be crowned our **Primary Model**, optimized for maximum yield in daily peace-time operations.
* **The "Uncertain/Noisy" World Weighting:** Simulates systemic data degradation, transmission latencies, or database corruptions where data is present but unreliable. Probability weights are shifted toward the right tail. The estimator that demonstrates the flattest degradation profile and higher $EP$ here will be selected as our **Robust Model (1st Backup)**, engineered to maintain portfolio stability under high data entropy.

### 2. Feature Dropping resistance
We simulate catastrophic infrastructure failures, such as third-party Credit Bureau API downtimes, severe server timeouts, or incomplete web-form submissions, by systematically masking entire features from our validation datasets. Missing values will be dynamically handled post-drop using an isolated imputation strategy (e.g., `SimpleImputer` leveraging training medians/modes).

Using a customized probabilistic weighting scheme tailored to information-loss scenarios, we map out the empirical expected profit under progressive variable omission:
$$EP = \sum_{j} Y_j \times \text{Profit}(\text{Dropped Scenario}_j)$$

This diagnostic isolates our third distinct architecture:
* **The "Missing Data" Failsafe model:** The estimator that maximizes expected profit when predictive features are missing will be selected as our **Failsafe Model (Backup 2)**. 
### 3. Feature Importance Stability under Noise
Before deploying our chosen estimators, we must ensure that their decision-making logic is structurally stable and does not suffer from "algorithmic hysteria" when input data degrades. We implement a geometric stress test to monitor how feature importance rankings and directionalities shift under progressive noise injection.

To evaluate stability, we must first isolate the $K$ most influential features for each architecture on clean data. Since the scaling of our variables is standardized using a `RobustScaler`, we can directly compare the magnitudes of their weights or importances:

* **Tree-Based Ensembles (RF & XGB):** We extract native impurity-based or gain-based feature importances ($I_i \ge 0$).
* **Linear Models (LR):** The importance of a feature is determined by the magnitude of its impact, regardless of the direction of risk. We establish the baseline ranking using the absolute values of the coefficients ($|w_i|$).

To evaluate structural decay, we compute the raw **Cosine Similarity** between the baseline importance vector ($v_{\text{base}}$) and the noisy importance vector ($v_{\text{noisy}}$) restricted to the Top-$K$ subspace:

$$\text{Stability}(v_{\text{base}}, v_{\text{noisy}}) = \frac{v_{\text{base}} \cdot v_{\text{noisy}}}{\|v_{\text{base}}\| \|v_{\text{noisy}}\|} \in [-1, 1]$$


* **For Logistic Regression (LR):** By keeping the original signs ($+$ or $-$) of the coefficients in the vectors, the similarity spans the full $[-1, 1]$ range. A drop towards $-1$ mathematically detects if the noise is severe enough to invert the model's interpretation of risk
* **For Tree-Based Models (RF & XGB):** Since tree importances are strictly non-negative, the similarity naturally spans the $[0, 1]$ range, where a drop towards $0$ indicates that the baseline dominant features have lost their predictive power.

This dual-nature metric isolates whether an architecture maintains a consistent backbone of core financial drivers or collapses into a chaotic state under progressive data corruption, providing the definitive validation of whether a model's interpretability can be trusted in production.

### 4. Compliant Post-Hoc Interpretability (SHAP & Assessment Reporting)

While global stability metrics validate the structural robustness of our architectures, regulatory compliance frameworks, such as Fair Lending acts, GDPR’s "Right to Explanation," and internal risk transparency guidelines, demand granular, actionable explainability for every credit decision. To bridge the gap between global model behavior and individual credit outcomes, we implement a post-hoc explainability pipeline across our model architectures:

*   **SHAP (SHapley Additive exPlanations):** Grounded in cooperative game theory, our pipeline utilizes SHAP to calculate Shapley values, ensuring a mathematically fair distribution of the credit score payoff across all input features. This allows us to decompose any automated credit decision into additive, feature-specific contributions. By quantifying the "contribution to risk" for every variable, we can transform abstract machine learning outputs into intuitive, legally defensible "adverse action" disclosures. This ensures that every automated rejection or credit limit adjustment is fully auditable, providing stakeholders and regulators with a clear trail of the financial drivers behind the decision.

*   **Automated Regulatory Assessment Reporting:** In alignment with international banking standards for transparency, our pipeline generates a comprehensive **Credit Decision Assessment Report** upon request. This document serves as a first justification for any credit revocation or application denial. The report includes:
    *   **Decision Attribution:** A breakdown of the primary variables (Key Risk Drivers) that exerted the most influence on the decision.
    *   **Visual Transparency:** A localized Waterfall Plot detailing how specific client behaviors (e.g., payment delays, utilization trends) shifted the risk score from the population mean toward the final decision.
    *   **Human-Readable Explanations:** A clear, non-technical explanation of the decision-making logic, ensuring that clients understand the specific financial behaviors that triggered the adverse action

> **Audit Readiness:** By standardizing the output into a formal PDF report, we reduce the burden on manual compliance reviews and satisfy the requirements for "Right to Explanation" under modern financial oversight regulations. 
> 
> Nonetheless, every rejected client has the right to request a formal review of their application as per applicable financial consumer protection laws (e.g., `Article 22 of the GDPR` in the European Union) and internal transparency policies, ensuring that a human expert assesses the specific circumstances and contestable elements of the automated decision.

### 5. Production Deployment Simulation
Finally, the three selected optimal model architectures are serialized and deployed into a production pipeline with a fully interactive web application hosted on **Hugging Face Spaces**. This application will simulate a live production environment, with the option to swap between models depending on the scenario


*Note for model selection: redundancy is not the goal, performance under different conditions is. If a model performs better than the rest in more than one scenario, then the same model can be chosen twice*

## 1. Imports

As always, we start by importing necessary libraries

In [31]:
import os
import copy
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy.special import logit, expit
import sklearn
import xgboost
import joblib
import datetime
import warnings
import shap
from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, Table, TableStyle, PageBreak
from reportlab.lib.enums import TA_RIGHT

And loading our best performing models, along with our model-ready datasets and previously designed **probability calibrating** and **custom value metric** functions

In [32]:
# Load datasets

train_input_raw = pd.read_csv("./data/splits/raw/train_input_raw.csv", index_col=0)
test_input_raw = pd.read_csv("./data/splits/raw/test_input_raw.csv", index_col=0)
train_input = pd.read_csv("./data/splits/preprocessed/train_input.csv", index_col=0)
train_target = pd.read_csv("./data/splits/preprocessed/train_target.csv", index_col=0)
test_input = pd.read_csv("./data/splits/preprocessed/test_input.csv", index_col=0)
test_target = pd.read_csv("./data/splits/preprocessed/test_target.csv", index_col=0)
train_EAD = pd.read_csv("./data/splits/EADs/train_EAD.csv", index_col=0)
test_EAD = pd.read_csv("./data/splits/EADs/test_EAD.csv", index_col=0)

# Load models
lr_model_1 = joblib.load('./models/LR_Model_1.joblib')
lr_model_2 = joblib.load('./models/LR_Model_2.joblib')
rf_model_1 = joblib.load('./models/RF_Model_1.joblib')
rf_model_2 = joblib.load('./models/RF_Model_2.joblib')
xgb_model_1 = joblib.load('./models/XGB_Model_1.joblib')
xgb_model_2 = joblib.load('./models/XGB_Model_2.joblib')

In [33]:
# Pre-calculate non-scaled EAD vectors for validation and test sets using original data before scaling
train_ead = train_EAD.values
test_ead = test_EAD.values

def calibrate_probabilities(p_raw, pi_train=0.2213, pi_real=0.003):
    """Calibrate raw probabilities to account for class imbalance between training and real-world distributions (Saerens et al. 2002)."""
    p_raw = np.array(p_raw).flatten()
    w_pos = pi_real / pi_train
    w_neg = (1 - pi_real) / (1 - pi_train)
    numerator = p_raw * w_pos
    denominator = numerator + (1 - p_raw) * w_neg
    return numerator / denominator

def custom_value_metric(y_true, y_pred, ead_vector, pi_train=0.2213, pi_real=0.003):
    """Custom value metric based on True Negatives, False Negatives, and True Positives, weighted by EAD and adjusted for class imbalance."""
    y_true = np.array(y_true).flatten()
    y_pred = np.array(y_pred).flatten()
    ead_vector = np.array(ead_vector).flatten()

    w_pos = pi_real / pi_train
    w_neg = (1 - pi_real) / (1 - pi_train)

    LGD = 0.70
    mitigation_cost_factor = 0.10
    gain_factor = 0.01

    tn_mask = (y_true == 0) & (y_pred == 0)
    fn_mask = (y_true == 1) & (y_pred == 0)
    tp_mask = (y_true == 1) & (y_pred == 1)

    tn_revenue = np.sum(np.maximum(ead_vector[tn_mask], 0) * gain_factor * w_neg)
    fn_loss = -np.sum(np.maximum(ead_vector[fn_mask], 0) * LGD * w_pos)
    tp_cost = -np.sum(np.maximum(ead_vector[tp_mask], 0) * mitigation_cost_factor * w_pos)

    return tn_revenue + fn_loss + tp_cost

In [34]:
os.makedirs('./tests', exist_ok=True)
os.makedirs('./data/tests', exist_ok=True)
os.makedirs('./tests/noise_injection', exist_ok=True)
os.makedirs('./tests/feature_dropping', exist_ok=True)
os.makedirs('./tests/similarity_under_noise', exist_ok=True)
os.makedirs('./tests/shap_analysis', exist_ok=True)

## 2. Monte Carlo Noise Injection

We inject progressive noise controlled by an intensity parameter $\sigma \in [0, 0.50]$ at a 0.05 step grid. 

To preserve the nature of each data type, the noise generation pipeline is divided into Continuous, Discrete and Categorical features:

* **Continuous Features (`LIMIT_BAL`, `PAY_AMT*`, `PAID_TO_REMAINING_*`, `CREDIT_UTIL*`):** We apply a localized **Gaussian perturbation**. For each continuous column, we calculate the bounded empirical spread between the 1.5th and 98.5th percentiles ($\text{scale} = P_{98.5} - P_{1.5}$). We then inject a zero-mean normal noise vector scaled by $\sigma$: 
  $$\epsilon \sim \mathcal{N}(0, (\sigma \times \text{scale})^2)$$
  to simulate progressive transaction reporting errors or estimation drifts.

* **Discrete Features (`PAY_SEP` to `PAY_APR`):**
  Since payment delay statuses are strictly ordered integers (ranging from $-2$ to $8$), adding continuous fractions would break the data structure. Instead, we use a **stochastic ordinal shift**. With a probability equal to $\sigma$, a row undergoes a $\pm 1$ step mutation (50/50 probability for $+1$ and $-1$). To maintain statistical realism, boundary enforcement constraints are hardcoded: if a status is at its absolute floor ($-2$) or ceiling ($8$), the shift is deterministically forced inward ($+1$ and $-1$ respectively). This emulates data-entry typos or minor micro-temporal lags in banking reporting systems.

By tracking the system across this multi-channel corruption grid, we evaluate both the financial decay rate via our **Custom Profit Metric** and the degradation of probability reliability via the **Brier Score Loss**.

The output is visualized through two separate plots for each model, mapping the progressive shifts in the mean metrics accompanied by a $\pm 2\sigma$ Monte Carlo confidence interval for every noise level $\sigma \in [0, 0.50]$.

---

As the time to run the entire test is highly dependent on the hardware, only its results will be given, along with the used code for reproducibility

Code:
```python
def calibrated_brier_score(y_true, p_raw, pi_train=0.2213, pi_real=0.003):  #  brier score metric with calibrated probabilities
    y_true = np.array(y_true).flatten()
    p_calibrated = calibrate_probabilities(p_raw, pi_train, pi_real)

    w_pos = pi_real / pi_train
    w_neg = (1 - pi_real) / (1 - pi_train)
    sample_weight = np.where(y_true == 1, w_pos, w_neg)

    return sklearn.metrics.brier_score_loss(
        y_true, p_calibrated, sample_weight=sample_weight
    )

# REDEFINE PREPROCESSOR (as in 03_Preprocessing_and_modeling)
numeric_and_ordinal = [col for col in train_input_raw.columns]

preprocessor = sklearn.compose.ColumnTransformer(
    transformers=[
        ('num', sklearn.preprocessing.RobustScaler(), numeric_and_ordinal)
    ],
    remainder='drop'
)

# DEFINE COLUMN TYPES (raw, not preprocessed)
raw_continuous_cols = [
    'LIMIT_BAL',
    'PAY_AMTSEP', 'PAY_AMTAUG', 'PAY_AMTJUL', 'PAY_AMTJUN', 'PAY_AMTMAY', 'PAY_AMTAPR',
    'PAID_TO_REMAINING_SEP', 'PAID_TO_REMAINING_AUG', 'PAID_TO_REMAINING_JUL', 'PAID_TO_REMAINING_JUN', 'PAID_TO_REMAINING_MAY', 'PAID_TO_REMAINING_APR',
    'CREDIT_UTIL_MEAN', 'CREDIT_UTIL_TREND'
]
raw_discrete_cols = ['PAY_SEP', 'PAY_AUG', 'PAY_JUL', 'PAY_JUN', 'PAY_MAY', 'PAY_APR']

# MAIN LOOP

np.random.seed(42)
sigmas = np.round(np.arange(0, 0.51, 0.05), 2)
n_iterations = 50   # iterations for every Fold (to average out the different noises) 
n_splits = 5    # CV splits

# Get folds
skf = sklearn.model_selection.StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
folds = list(skf.split(train_input_raw, train_target))

models = {
    "LR_1": lr_model_1,
    "LR_2": lr_model_2,
    "RF_1": rf_model_1,
    "RF_2": rf_model_2,
    "XGB_1": xgb_model_1,
    "XGB_2": xgb_model_2
}

thresholds = joblib.load("./models/decision_thresholds.joblib")

all_results = {}

for model_name, model in models.items(): # For every model

    mean_profit, std_profit = [], []
    mean_brier, std_brier  = [], []

    for sigma in sigmas: # For every sigma value

        all_profits, all_briers = [], []  # accumulate raw: fold x n_iterations

        for train_idx, val_idx in folds: # For every fold

            # Get raw fold
            X_train_raw = train_input_raw.iloc[train_idx].copy()
            X_val_raw = train_input_raw.iloc[val_idx].copy()
            y_train = train_target.iloc[train_idx].values.ravel()
            y_val = train_target.iloc[val_idx].values.ravel()
            ead_val = np.array(train_EAD)[val_idx]

            # Preprocess avoiding data leakeage on validation fold
            coverage_cols = [f"PAID_TO_REMAINING_{m}" for m in ["SEP","AUG","JUL","JUN","MAY","APR"]]
            for col in coverage_cols:
                X_train_raw[col] = X_train_raw[col].clip(upper=2.0)
                X_val_raw[col] = X_val_raw[col].clip(upper=2.0)

            X_train_raw["CREDIT_UTIL_MEAN"] = X_train_raw["CREDIT_UTIL_MEAN"].clip(0.0, 1.5)
            X_val_raw["CREDIT_UTIL_MEAN"]  = X_val_raw["CREDIT_UTIL_MEAN"].clip(0.0, 1.5)

            X_train_raw["CREDIT_UTIL_TREND"] = X_train_raw["CREDIT_UTIL_TREND"].clip(-0.3, 0.3)
            X_val_raw["CREDIT_UTIL_TREND"]  = X_val_raw["CREDIT_UTIL_TREND"].clip(-0.3, 0.3)

            pay_amt_cols = [f"PAY_AMT{m}" for m in ["SEP","AUG","JUL","JUN","MAY","APR"]]
            for col in pay_amt_cols:
                fold_q99 = X_train_raw[col].quantile(0.99)
                X_train_raw[col] = X_train_raw[col].clip(upper=fold_q99)
                X_val_raw[col]  = X_val_raw[col].clip(upper=fold_q99)

            local_preprocessor = sklearn.base.clone(preprocessor)
            X_train_scaled = local_preprocessor.fit_transform(X_train_raw)
            # X_val_raw NOT scaled here, but after adding noise

            X_train = pd.DataFrame(X_train_scaled, columns=numeric_and_ordinal, index=X_train_raw.index)

            # Train the model on scaled train data
            model_fold = sklearn.base.clone(model)
            model_fold.fit(X_train, y_train)

            # Monte Carlo: inject noise -> preprocess -> predict
            for _ in range(n_iterations):

                X_noisy_raw = X_val_raw.copy()  # work on RAW

                if sigma != 0:

                    # CONTINUOUS: Gaussian noise on fold distribution
                    for col in raw_continuous_cols:
                        lower, upper = np.percentile(X_noisy_raw[col], [1.5, 98.5])
                        scale = upper - lower
                        noise = np.random.normal(0, sigma * scale, size=len(X_noisy_raw))
                        X_noisy_raw[col] += noise

                    # DISCRETE (PAY_*): ordinal shift ±1 with boundary enforcement
                    for col in raw_discrete_cols:
                        mask = np.random.rand(len(X_noisy_raw)) < sigma
                        if mask.any():
                            deltas = np.random.choice([-1, 1], size=mask.sum())
                            current = X_noisy_raw.loc[X_noisy_raw.index[mask], col].values
                            deltas = np.where(current == 8, -1, deltas)
                            deltas = np.where(current == -2, 1, deltas)
                            X_noisy_raw.loc[X_noisy_raw.index[mask], col] = (current + deltas).astype(X_noisy_raw[col].dtype)

                # Preprocess noisy data (test)
                X_noisy_scaled = local_preprocessor.transform(X_noisy_raw)
                X_noisy = pd.DataFrame(X_noisy_scaled, columns=numeric_and_ordinal, index=X_val_raw.index)

                # Predict
                y_proba = model_fold.predict_proba(X_noisy)[:, 1]
                y_pred = (calibrate_probabilities(y_proba) >= thresholds[model_name]).astype(int)

                # Append RAW metrics (no per-fold averaging)
                all_profits.append(custom_value_metric(y_val, y_pred, ead_val) / len(y_val))
                all_briers.append(calibrated_brier_score(y_val, y_proba))

        # Append final model metrics: mean/std on all fold*n_iterations
        mean_profit.append(np.mean(all_profits))
        std_profit.append(np.std(all_profits))
        mean_brier.append(np.mean(all_briers))
        std_brier.append(np.std(all_briers))

        print(f"{model_name}: {sigma:.2f} sigma finished")

    # Save model performances
    all_results[model_name] = {
        "sigmas": np.array(sigmas),
        "mean_profit": np.array(mean_profit),
        "std_profit": np.array(std_profit),
        "mean_brier": np.array(mean_brier),
        "std_brier": np.array(std_brier)
    }


### PLOTS

lr_models = ["LR_1", "LR_2"]
rf_models = ["RF_1", "RF_2"]
xgb_models = ["XGB_1", "XGB_2"]

groups = [
    ("LR", lr_models),
    ("RF", rf_models),
    ("XGB", xgb_models)
]

# PROFITS
fig, axes = plt.subplots(3, 2, figsize=(21, 34))

for i, (name, model) in enumerate(groups):

    for j in range(2): # 3x2 grid

        model_name = model[j]
        res = all_results[model_name]

        mean, std = res["mean_profit"], res["std_profit"]

        axes[i, j].plot(sigmas, mean, color="blue")

        axes[i, j].fill_between(
            sigmas,
            mean - 2 * std,
            mean + 2 * std,
            color="blue",
            alpha=0.15
        )

        axes[i, j].set_title(f"{name} - {model_name}")
        axes[i, j].grid(True, linestyle=":")
        axes[i, j].set_ylim(10, 15)

        axes[i, j].xaxis.set_major_locator(ticker.MultipleLocator(0.05))
        axes[i, j].yaxis.set_major_locator(ticker.MultipleLocator(0.2))

        axes[i, j].set_xlabel("Sigma")
        axes[i, j].set_ylabel("Profit")

plt.suptitle("Noise injection - Profit Metric", fontsize=20, y=0.96)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("./tests/noise_injection/Profits.png")

# BRIERS
fig, axes = plt.subplots(3, 2, figsize=(21, 34))

for i, (name, model) in enumerate(groups):

    for j in range(2): # 3x2 grid

        model_name = model[j]
        res = all_results[model_name]

        mean = np.array(res["mean_brier"])
        std = np.array(res["std_brier"])

        axes[i, j].plot(sigmas, mean, color="red")

        axes[i, j].fill_between(
            sigmas,
            mean - 2 * std,
            mean + 2 * std,
            color="red",
            alpha=0.15
        )

        axes[i, j].set_title(f"{name} - {model_name}")
        axes[i, j].grid(True, linestyle=":")
        axes[i, j].set_ylim(0.0028, 0.0033)

        axes[i, j].xaxis.set_major_locator(ticker.MultipleLocator(0.05))
        axes[i, j].yaxis.set_major_locator(ticker.MultipleLocator(0.00005))

        axes[i, j].set_xlabel("Sigma")
        axes[i, j].set_ylabel("Brier")

plt.suptitle("Noise injection - Brier Score", fontsize=20, y=0.96)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("./tests/noise_injection/Briers.png")


# Save results
joblib.dump(all_results, "./tests/noise_injection/results.pkl")
```

---

Results:

![Profit Graph (in /tests/noise_injection/Profits.png)](./tests/noise_injection/Profits.png)

![Brier Graph (in /tests/noise_injection/Briers.png)](./tests/noise_injection/Briers.png)

As visible from the results, Logistic Regression (`LR_1`, as `LR_2` exhibits completely different behavior) tends to resist more under added noise, while starting to degrade only at extreme noise values.

Random Forests, on the other hand, while starting at a higher profit baseline, quickly settle to an equilibrium point just above $\$13.00/cust$

XGBoost models instead (`XGB_2` in particular) exhibits outstanding performance at higher noises, positioning itself over $\$0.05/cust$ over the other architectures

To confirm these assumptions, further tests need to be done. Instead of basing the decision on single graphs, a more realistic test will be made to validate our readings by creating two separate scenarios: a **Normal/Stable world** (little to no noise, minor reporting errors) and an **Uncertain/Unstable world** (non-negligible noise, reporting errors, service malfunctions).

Obviously, even in a perfect world, noise is always present. We can then construct a simple profit-sigma weighting function to calculate an estimated profit based on the number of instances with a certain level of noise.

The expected economic value under stochastically weighted risk profiles is formalized as a discrete probability distribution over our $\sigma$ noise steps:

$$Final\ Profit = \sum_{k} \text{Profit}(\sigma_k) \times P(\sigma_k)$$

Where:
* $\text{Profit}(\sigma_k)$ is the financial return generated by the model at a specific noise level $\sigma_k$.
* $P(\sigma_k)$ is the probability (or weight) of that specific noise level occurring in the chosen scenario, ensuring that $\sum P(\sigma_k) = 1.0$ $(100\%)$

**Scenario Weights Definition**

1. **Normal World (Stable Market):** Most data points are clean ($\sigma = 0.00$ to $0.05$), with a tiny decay tail representing rare operational hiccups.
   
   -> 85% probability of $0.00 \sigma$ noise, 10% of $0.05 \sigma$ and  5% of $0.10 \sigma$
2. **Uncertain World (Volatile Market):** The probability shifts towards medium and high levels of noise ($\sigma \ge 0.15$), simulating severe systemic crises, data pipeline lags, or data corruption.
   
    -> 10% probability of $0.00 \sigma$ noise, 15% of $0.05 \sigma$, 20% of $0.10 \sigma$, 15% of $0.15 \sigma$, 10% of $0.20 \sigma$, 8% of $0.25 \sigma$, 7% of $0.30 \sigma$, 5% of $0.35 \sigma$, 5% of $0.40 \sigma$, 3% of $0.45 \sigma$ and 2% of $0.50 \sigma$

In [35]:
# Load results
all_results = joblib.load("./tests/noise_injection/results.pkl")

sigmas = all_results[list(all_results.keys())[0]]["sigmas"]

# Probabiltistic weights (sum = 1)
weights_nominal = np.array([0.85, 0.10, 0.05, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00])
weights_uncertain = np.array([0.10, 0.15, 0.20, 0.15, 0.10, 0.08, 0.07, 0.05, 0.05, 0.03, 0.02])

print("=========== EP METRICS (Normal and Uncertain worlds) ===========")
print("-" * 65)

ep_summary = {}

for model_name, metrics in all_results.items():
    mean_profits = metrics["mean_profit"]
    
    # Scalar product to calculate EPs
    ep_nominal = np.dot(mean_profits, weights_nominal)
    ep_uncertain = np.dot(mean_profits, weights_uncertain)
    
    # Save model perormance
    ep_summary[model_name] = {
        "EP_Nominal": ep_nominal,
        "EP_Uncertain": ep_uncertain
    }
    
    # Print
    print(f"Model {model_name:5} | EP Nominal: ${ep_nominal:6.2f}/cust | EP Uncertain: ${ep_uncertain:6.2f}/cust")

print("-" * 65)

# Choose best for each case
primary_model = max(ep_summary, key=lambda k: ep_summary[k]["EP_Nominal"])
robust_model = max(ep_summary, key=lambda k: ep_summary[k]["EP_Uncertain"])

# Print
print(f"- PRIMARY MODEL (Best in Normal World): {primary_model} (${ep_summary[primary_model]['EP_Nominal']:.2f}/cust)")
print(f"- ROBUST MODEL (Best in Uncertain World): {robust_model} (${ep_summary[robust_model]['EP_Uncertain']:.2f}/cust)")

=========== EP METRICS (Normal and Uncertain worlds) ===========
-----------------------------------------------------------------
Model LR_1  | EP Nominal: $ 13.38/cust | EP Uncertain: $ 13.28/cust
Model LR_2  | EP Nominal: $ 13.27/cust | EP Uncertain: $ 13.02/cust
Model RF_1  | EP Nominal: $ 13.42/cust | EP Uncertain: $ 13.21/cust
Model RF_2  | EP Nominal: $ 13.39/cust | EP Uncertain: $ 13.21/cust
Model XGB_1 | EP Nominal: $ 13.36/cust | EP Uncertain: $ 13.24/cust
Model XGB_2 | EP Nominal: $ 13.38/cust | EP Uncertain: $ 13.27/cust
-----------------------------------------------------------------
- PRIMARY MODEL (Best in Normal World): RF_1 ($13.42/cust)
- ROBUST MODEL (Best in Uncertain World): LR_1 ($13.28/cust)


The empirical results from the Monte Carlo simulation over the two synthetic market conditions (**Nominal World** vs. **Uncertain World**) provide further indication for our model selection strategy. 

#### 1. The Structural advantage of Random Forest
Our experimental results confirm a significant structural vulnerability in the Random Forest architecture under covariate shift and out-of-distribution training noise:
* **Nominal Performance:** On clean data, `RF_1` yields the maximum return out of all the candidates at $\$13.39/cust$
* **Uncertain Regime:** Under progressive noise injection, RF suffers the worst financial drawdown out of all tested models, while "only" losing about **1.6%** of its economic value
* **Underlying Mechanism:** Because the injected noise is distributed across the entire feature space rather than being isolated to dominant features, individual unregularized decision trees within the ensemble, which naturally split on secondary, non-defining variables, experience widespread prediction flips. Lacking a global shrinkage parameter or structural L1/L2 constraints, the cumulative voting mechanism amplify these local errors, thoroughly corrupting the ensemble's final output.

#### 2. Logistic Regression as a Robust Financial Shield
While linear models exhibit a lower profit ceiling in stable markets due to their rigid mathematical assumptions, they display exceptional resilience to data degradation:
* **Uncertain Regime:** `LR_1` emerges as a robust candidate for extremely noisy scenarios, retaining **$13.28 per customer** in the Uncertain World, a minor performance decay of only 0.7% compared to the 'Realistic Scenario'.
* **Underlying Mechanism:** Severe L1 Lasso regularization (`C=0.001`) enforces strict **structural parsimony** by limiting or directly zeroing out coefficients associated with secondary, noisy features. When noise is injected globally, the model's low-capacity linear boundary prevents it from chasing high-frequency background noise features.

*(Looking back at the total number of active features between `LR_1` and `LR_2`, we can remember how the first candidate only had 9/21 total active features, with an important weight decrease after the top ones, while `LR_2` maintained almost every feature and failed to perform, possibly due to their intrinsic independent-weights architecture)*

#### 3. Gradient Boosting settles as a middleground option (`XGB_2`)
* **Nominal Performance:** It places himself as the second best architecture in the Nominal World (**$13.38 per customer**)
* **Uncertain Regime:** Delivers the second best performance, achieving a solid **$13.27 per customer** even under high stress and noisy scenarios

### Final Architectural Decision

The enterprise credit scoring engine will deploy a **Dual-Engine Switching Architecture**:
1. **Primary Model (`RF_1`):** Active during normal operating conditions (defined by stable data pipeline schemas and standard macroeconomic indicators) to capture high-order non-linear risk interactions and maximize operational profit.
2. **Robust Model (1st Backup) (`LR_1`):** Hot-swapped into active scoring the moment data integrity indicators (e.g., real-time feature drift, missing value spikes) or market volatility indexes exceed a predefined risk tolerance threshold, ensuring a solid fallback under extreme stress while also remaining competitive under normal conditions.

(even in the worst case, the chosen "Robust" model performs better than the "all 0s" baseline, so the use of Machine Learning is defendable)

## 3. Feature Dropping Resistance & Fail-Safe Selection

While global noise tests evaluate how models handle *corrupted* data, pipeline failures in production often manifest as *missing* data. Incomplete customer application sheets, API timeouts, or database sync failures frequently force models to score applicants with missing variables. 

To select our **Failsafe Model (2nd Backup)**, we subject our architectures to a progressive feature dropping stress test. The experiment simulates a total pipeline breakdown by randomly dropping between $1$ and $10$ (50%) features from the input matrix, using simple **median imputation** as the baseline recovery strategy. To minimize variance, we run $100$ Monte Carlo iterations for each level of feature omission.

---

Code:
```python
def calibrated_brier_score(y_true, p_raw, pi_train=0.2213, pi_real=0.003):  #  brier score metric with calibrated probabilities
    y_true = np.array(y_true).flatten()
    p_calibrated = calibrate_probabilities(p_raw, pi_train, pi_real)

    w_pos = pi_real / pi_train
    w_neg = (1 - pi_real) / (1 - pi_train)
    sample_weight = np.where(y_true == 1, w_pos, w_neg)

    return sklearn.metrics.brier_score_loss(
        y_true, p_calibrated, sample_weight=sample_weight
    )

# DEFINE COLUMN TYPES
numeric_and_ordinal = [col for col in train_input_raw.columns]

# REDEFINE PREPROCESSOR (as in 03_Preprocessing_and_modeling)
preprocessor = sklearn.compose.ColumnTransformer(
    transformers=[
        ('num', sklearn.preprocessing.RobustScaler(), numeric_and_ordinal) 
    ],
    remainder='drop'
)

all_raw_features = train_input_raw.columns.tolist()

# MAIN LOOP

np.random.seed(42)
drop_counts = np.arange(0, len(all_raw_features)//2 + 1, 1) # from 0 to 50% dropped columns
n_iter = 100    # iterations for every Fold (to average out the different features dropped) 
n_splits = 5    # CV splits

# Get folds
skf = sklearn.model_selection.StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
folds = list(skf.split(train_input_raw, train_target))

# Models dict
models = {
    "LR_1": lr_model_1, "LR_2": lr_model_2,
    "RF_1": rf_model_1, "RF_2": rf_model_2,
    "XGB_1": xgb_model_1, "XGB_2": xgb_model_2
}

thresholds = joblib.load("./models/decision_thresholds.joblib")

dropping_results = {}

# For every model, do n_iter random feature dropping cycles with Cross Validation to evaluate performance
for model_name, model in models.items():

    mean_profit, std_profit = [], []
    mean_brier, std_brier  = [], []

    for k in drop_counts: # For every number of features to drop

        all_profits, all_briers = [], []  # accumulate raw: fold x n_iter

        for train_idx, val_idx in folds:

            # Get raw fold
            X_train_raw = train_input_raw.iloc[train_idx].copy()
            X_val_raw = train_input_raw.iloc[val_idx].copy()
            y_train = train_target.iloc[train_idx].values.ravel()
            y_val = train_target.iloc[val_idx].values.ravel()
            ead_val = np.array(train_EAD)[val_idx]

            # Preprocess avoiding data leakeage on validation fold
            coverage_cols = [f"PAID_TO_REMAINING_{m}" for m in ["SEP","AUG","JUL","JUN","MAY","APR"]]
            for col in coverage_cols:
                X_train_raw[col] = X_train_raw[col].clip(upper=2.0)
                X_val_raw[col] = X_val_raw[col].clip(upper=2.0)

            X_train_raw["CREDIT_UTIL_MEAN"] = X_train_raw["CREDIT_UTIL_MEAN"].clip(0.0, 1.5)
            X_val_raw["CREDIT_UTIL_MEAN"]  = X_val_raw["CREDIT_UTIL_MEAN"].clip(0.0, 1.5)

            X_train_raw["CREDIT_UTIL_TREND"] = X_train_raw["CREDIT_UTIL_TREND"].clip(-0.3, 0.3)
            X_val_raw["CREDIT_UTIL_TREND"]  = X_val_raw["CREDIT_UTIL_TREND"].clip(-0.3, 0.3)

            pay_amt_cols = [f"PAY_AMT{m}" for m in ["SEP","AUG","JUL","JUN","MAY","APR"]]
            for col in pay_amt_cols:
                fold_q99 = X_train_raw[col].quantile(0.99)
                X_train_raw[col] = X_train_raw[col].clip(upper=fold_q99)
                X_val_raw[col]  = X_val_raw[col].clip(upper=fold_q99)

            # Fit imputers and preprocessor on the CLEAN TRAIN of the fold
            local_preprocessor = sklearn.base.clone(preprocessor)
            X_train_scaled = local_preprocessor.fit_transform(X_train_raw)

            imputer_num = sklearn.impute.SimpleImputer(strategy="median").fit(X_train_raw[numeric_and_ordinal])

            X_train = pd.DataFrame(X_train_scaled, columns=numeric_and_ordinal, index=X_train_raw.index)

            # Train on clean train fold
            model_fold = sklearn.base.clone(model)
            model_fold.fit(X_train, y_train)

            # Monte Carlo iterations to average out the randomness of dropped features
            for _ in range(n_iter if k > 0 else 1):
                X_dropped_raw = X_val_raw.copy()

                if k > 0:
                    features_to_remove = np.random.choice(all_raw_features, size=k, replace=False)
                    X_dropped_raw[features_to_remove] = np.nan

                X_dropped_raw[numeric_and_ordinal] = imputer_num.transform(X_dropped_raw[numeric_and_ordinal])

                X_noisy_scaled = local_preprocessor.transform(X_dropped_raw)
                X_noisy = pd.DataFrame(X_noisy_scaled, columns=numeric_and_ordinal, index=X_val_raw.index)

                y_proba = model_fold.predict_proba(X_noisy)[:, 1]
                y_pred = (calibrate_probabilities(y_proba) >= thresholds[model_name]).astype(int)

                # Append RAW metrics (no per-fold averaging)
                all_profits.append(custom_value_metric(y_val, y_pred, ead_val) / len(y_val))
                all_briers.append(calibrated_brier_score(y_val, y_proba))

        # Mean and dev.st on all (fold*n_iter, or only fold if k==0)
        mean_profit.append(np.mean(all_profits))
        std_profit.append(np.std(all_profits))
        mean_brier.append(np.mean(all_briers))
        std_brier.append(np.std(all_briers))

        print(f"{model_name}: {k} random features dropped finished")

    # Save final model performance in the results dictionary
    dropping_results[model_name] = {
        "dropped_counts": np.array(drop_counts),
        "mean_profit": np.array(mean_profit),
        "std_profit": np.array(std_profit),
        "mean_brier": np.array(mean_brier),
        "std_brier": np.array(std_brier)
    }


### PLOTS

lr_models = ["LR_1", "LR_2"]
rf_models = ["RF_1", "RF_2"]
xgb_models = ["XGB_1", "XGB_2"]

groups = [
    ("LR", lr_models),
    ("RF", rf_models),
    ("XGB", xgb_models)
]

# PROFITS
fig, axes = plt.subplots(3, 2, figsize=(21, 34))

for i, (name, model_pair) in enumerate(groups):
    for j in range(2): # 3x2 grid

        model_name = model_pair[j]
        res = dropping_results[model_name]

        mean, std = res["mean_profit"], res["std_profit"]

        axes[i, j].plot(drop_counts, mean, color="blue")

        axes[i, j].fill_between(
            drop_counts,
            mean - 2 * std,
            mean + 2 * std,
            color="blue",
            alpha=0.15
        )
        
        axes[i, j].set_title(f"{name} - {model_name} (Feature Dropping)")
        axes[i, j].grid(True, linestyle=":")
        axes[i, j].set_ylim(10, 15)

        axes[i, j].xaxis.set_major_locator(ticker.MultipleLocator(1))
        axes[i, j].yaxis.set_major_locator(ticker.MultipleLocator(0.2))

        axes[i, j].set_xlabel("Number of Randomly Dropped Features")
        axes[i, j].set_ylabel("Profit")

plt.suptitle("Feature Dropping - Profit Metric", fontsize=20, y=0.96)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("./tests/feature_dropping/Profits.png")

# BRIERS

fig, axes = plt.subplots(3, 2, figsize=(21, 34))

for i, (name, model_pair) in enumerate(groups):
    for j in range(2): # 3x2 grid

        model_name = model_pair[j]
        res = dropping_results[model_name]

        mean, std = res["mean_brier"], res["std_brier"]

        axes[i, j].plot(drop_counts, mean, color="red")
        axes[i, j].fill_between(
            drop_counts,
            mean - 2 * std,
            mean + 2 * std,
            color="red",
            alpha=0.15
        )
        
        axes[i, j].set_title(f"{name} - {model_name} (Feature Dropping)")
        axes[i, j].grid(True, linestyle=":")
        axes[i, j].set_ylim(0.0028, 0.0033)
        
        axes[i, j].xaxis.set_major_locator(ticker.MultipleLocator(1))
        axes[i, j].yaxis.set_major_locator(ticker.MultipleLocator(0.00005))

        axes[i, j].set_xlabel("Number of Randomly Dropped Features")
        axes[i, j].set_ylabel("Brier Score")

plt.suptitle("Feature Dropping - Brier Score", fontsize=20, y=0.96)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("./tests/feature_dropping/Briers.png")
plt.show()

# Save results
joblib.dump(dropping_results, "./tests/feature_dropping/results.pkl")
```

---

Results:

![Profit Graph (in /tests/feature_dropping/Profits.png)](./tests/feature_dropping/Profits.png)

![Brier Graph (in /tests/feature_dropping/Briers.png)](./tests/feature_dropping/Briers.png)

In contrast to the noise injection stress test, where Random Forest suffered the worst economic collapse, the empirical results under progressive feature dropping reveal a complete inversion of performance. **Random Forest (RF)** models exhibit outstanding robustness in scenarios with few missing features, though they are eventually surpassed by simpler linear models (e.g., `LR_1`) as the degree of feature missingness increases.

The diverging performance under missing data highlights a fundamental split in how these mathematical architectures process and rely on their input space:

### 1. The Advantage of the Boosting Chain in Noise Resilience (XGBoost)
The contrast between XGBoost's high resilience to noise and its sudden vulnerability to feature dropping exposes a fundamental mathematical truth about how boosting handles different types of data degradation:
* **Superior Noise Resilience (The Self-Canceling Effect):** 
  Under the noise injection test, perturbations follow a normal distribution centered at zero ($\mu=0$). Because XGBoost is an additive ensemble that sums the predictions of hundreds of small, sequential trees ($f(x) = \sum h_t(x)$), these high-frequency random fluctuations tend to **cancel each other out** across the boosting chain. The law of large numbers shields the model: positive and negative noise components average out, allowing XGBoost to maintain strong performance even at higher noise levels.
* **Vulnerability to Feature Dropping:** 
  Unlike distributed noise, dropping a feature represents a systematic, localized loss of critical information. When a high-gain, dominant variable (like repayment history) is omitted and flatlined with its median, there is no random fluctuation to "average out." 
  Since XGBoost trains trees sequentially to fit the gradient of the loss function, the very first trees in the chain rely heavily on these high-gain features to make primary risk partitions. All downstream trees are mathematically optimized *under the assumption* that those primary partitions were executed correctly. Removing the target feature blindfolds the primary trees, forcing them to output massive, distorted errors. Because subsequent trees cannot "route around" this missing step, the entire downstream error-correction cascade collapses, causing a dramatic drop in financial performance.

### 2. The Rigidity of Linear Coefficients (Logistic Regression)
Logistic Regression relies on a static, global weighted sum of all inputs ($z = \beta_0 + \beta_1 x_1 + \dots + \beta_n x_n$).
* **Distortion via Median Imputation:** When a feature is dropped and replaced with its median, we artificially collapse its variance to zero for those observations. Combined with Random Forest's higher profit baseline, this prevents linear models from outperforming tree ensembles in low-to-moderate missingness settings.
* **Mathematical Mismatch:** The model is forced to apply its active coefficient ($\beta_i$) to a flat, non-informative median value. Because a linear model lacks the ability to "route" its decisions around missing data, this distortion directly shifts the log-odds of default, leading to systematic risk miscalibration.
* **Regained Competitiveness at High Missingness Rates:** At higher feature dropping rates (e.g., 5 or more missing features), linear models outperform ensemble methods. Having less than half of the entire feature set active at each prediction (`LR_1`) reduces the overall impact of feature dropping by nearly half, allowing the simple structure of linear models to stabilize predictions.

### 3. The Robustness of Independent Bagging (Random Forest)
* **Dampening Local Errors via Bootstrap Averaging:** Each tree is fitted on an independent bootstrap resample. A prediction error introduced by a median-imputed value in one tree is not propagated to the rest of the ensemble; instead, it is diluted by averaging across hundreds of independently resampled estimators as a variance-reduction effect.
* **Interaction with Pre-Scaled, Winsorized Inputs:** Because all features are RobustScaler-transformed and capped at training time, median imputation at inference substitutes a "central" value on an already-bounded scale. Since tree splits are threshold-based rather than coefficient-based, a median-imputed value tends to fall into a plausible decision region rather than distorting a fixed linear combination.

In [36]:
# Load results
dropping_results = joblib.load("./tests/feature_dropping/results.pkl")

print("TOP 3 MODELS FOR EVERY DROP LEVEL")
print("=" * 35)

for k in range(1, 11): # For every level of dropped features (1 to 10)
    rankings = []
    for model_name, metrics in dropping_results.items():
        idx = np.where(metrics["dropped_counts"] == k)[0]
        if len(idx) > 0:
            i = idx[0]
            rankings.append((model_name, metrics["mean_profit"][i], metrics["std_profit"][i])) # Save model name, mean profit, and std profit for the current drop level
            
    # Order for descending profit
    rankings.sort(key=lambda x: x[1], reverse=True)
    
    # Format the row by inserting mean and 2*sigma
    top_str = " | ".join([f"{mod} ({mu:.2f} ± {2*sigma:.2f} $/cust)" for mod, mu, sigma in rankings[:3]])
    print(f"{k} missing features -> {top_str}")

TOP 3 MODELS FOR EVERY DROP LEVEL
1 missing features -> RF_1 (13.40 ± 0.70 $/cust) | RF_2 (13.38 ± 0.69 $/cust) | LR_1 (13.36 ± 0.67 $/cust)
2 missing features -> RF_1 (13.38 ± 0.73 $/cust) | RF_2 (13.36 ± 0.71 $/cust) | LR_1 (13.34 ± 0.68 $/cust)
3 missing features -> RF_1 (13.35 ± 0.74 $/cust) | RF_2 (13.34 ± 0.74 $/cust) | XGB_2 (13.33 ± 0.71 $/cust)
4 missing features -> RF_2 (13.32 ± 0.76 $/cust) | RF_1 (13.31 ± 0.77 $/cust) | LR_1 (13.30 ± 0.74 $/cust)
5 missing features -> RF_1 (13.30 ± 0.81 $/cust) | RF_2 (13.30 ± 0.75 $/cust) | LR_1 (13.28 ± 0.77 $/cust)
6 missing features -> RF_1 (13.29 ± 0.80 $/cust) | LR_1 (13.28 ± 0.77 $/cust) | RF_2 (13.27 ± 0.82 $/cust)
7 missing features -> LR_1 (13.25 ± 0.77 $/cust) | RF_2 (13.25 ± 0.80 $/cust) | XGB_1 (13.25 ± 0.79 $/cust)
8 missing features -> LR_1 (13.24 ± 0.82 $/cust) | RF_1 (13.23 ± 0.80 $/cust) | RF_2 (13.23 ± 0.81 $/cust)
9 missing features -> LR_1 (13.23 ± 0.81 $/cust) | RF_2 (13.21 ± 0.84 $/cust) | RF_1 (13.21 ± 0.85 $/cust)
1

And with this last numeric test, our assumption is locked in: **Random Forests (`RF_1` in particular) outperforms XGB and LR models in all moderate feature-missing cases (i.e. less than 6 missing)**, while at higher drop rates **Logistic Regression (`LR_1` in particular)** manitains the lead when std-adjusted

(in all cases, the chosen fitted model performs better than the "all 0s" baseline, so the use of Machine Learning is defendable again)

We then choose our **Failsafe model (2nd Backup)** to be `LR_1`, completing the circle and confirming our operative model lineup:
- `RF_1` as our **Primary Model**
- `LR_1` as our noise-resistant **Robust Model (1st Backup)**
- `LR_1` as our information-loss **Failsafe Model (2nd Backup)** with **6 or more** missing features.

## 4. Feature Importance Stability under noise

To further validate our test results and ensure the operational robustness of our models, we evaluate the stability of top features when models are trained on noisy data.

This protocol ensures that the current feature importances are not chosen randomly by the estimators and possess genuine predictive power. If a model suddenly flips its most important features under minimal noise, it indicates weight instability and over-optimization on the validation set, signaling that the architecture cannot be fully trusted in production.

To calculate feature stability over this wide range of model architectures (Logistic Regression, Random Forest, Gradient Boosting), we implement the cosine similarity stress test framework described in the first section (`Key Tests and Operational Workflow`), where feature coefficients (LR) and feature importances (RF & XGB), stored before and after noise injection, are used to calculate cosine similarity between the two at each level of added noise to the train dataset

> **Methodological note:** this test differs from the Noise Injection in
> Section 2. There, noise is injected only on the validation set, with the
> scaler fitted on clean train data (simulating a corrupted inference
> pipeline in production). Here, noise is injected **before** training, and
> the scaler is refitted on the already-corrupted data (simulating a
> contaminated training set upstream, e.g. from a compromised data
> integration). The two scenarios answer different questions: "does the
> model degrade if the data it receives is dirty?" (Section 2) vs. "does
> the model learn different features if the data it's trained on is dirty?"
> (this section).

---

As with the previous tests, due to the highly hardware-dependent nature of these tests, only the results along with the used code are left

Code:
```python
# REDEFINE PREPROCESSOR (as in 03_Preprocessing_and_modeling)
numeric_and_ordinal = [col for col in train_input_raw.columns]

preprocessor = sklearn.compose.ColumnTransformer(
    transformers=[
        ('num', sklearn.preprocessing.RobustScaler(), numeric_and_ordinal)
    ],
    remainder='drop'
)

# DEFINE COLUMN TYPES (raw, not preprocessed)
raw_continuous_cols = [
    'LIMIT_BAL',
    'PAY_AMTSEP', 'PAY_AMTAUG', 'PAY_AMTJUL', 'PAY_AMTJUN', 'PAY_AMTMAY', 'PAY_AMTAPR',
    'PAID_TO_REMAINING_SEP', 'PAID_TO_REMAINING_AUG', 'PAID_TO_REMAINING_JUL', 'PAID_TO_REMAINING_JUN', 'PAID_TO_REMAINING_MAY', 'PAID_TO_REMAINING_APR',
    'CREDIT_UTIL_MEAN', 'CREDIT_UTIL_TREND'
]
raw_discrete_cols = ['PAY_SEP', 'PAY_AUG', 'PAY_JUL', 'PAY_JUN', 'PAY_MAY', 'PAY_APR']

# MAIN LOOP

np.random.seed(42)
sigmas = np.round(np.arange(0, 0.51, 0.05), 2)
n_iterations = 50   # iterations for every Fold (to average out the different noises) 
top_ks = [5, 10]  # Top K importances to calcualte cosine similarity for (going further, feature weights tend to flatten out, so any value greater than this would simply add noise to the simlarity function)

models = {
    "LR_1": lr_model_1,
    "LR_2": lr_model_2,
    "RF_1": rf_model_1,
    "RF_2": rf_model_2,
    "XGB_1": xgb_model_1,
    "XGB_2": xgb_model_2
}

all_results = {}

# HELPERS

def get_importances(model):
    """
    Get feature importances/coefficients from a model
    - LR → signed coef_[0] (keep original +/-)
    - RF/XGB → feature_importances_ (strictly +)
    """
    if hasattr(model, 'coef_'):
        return model.coef_[0].copy()
    else:
        return model.feature_importances_.copy()

def cosine_similarity(a, b):
    """Calculate cosine similarity from the formula"""
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom > 0 else 0.0

def inject_noise_train(X_raw, sigma):
    """Inject noise on raw training data (same protocol as in the stress test)"""
    X = X_raw.copy()
    if sigma == 0:
        return X
    # Continuous
    for col in raw_continuous_cols:
        lower, upper = np.percentile(X[col], [1.5, 98.5])
        X[col] += np.random.normal(0, sigma * (upper - lower), size=len(X)) # Add normal noise based on sigma and the range
    # Discrete
    for col in raw_discrete_cols:
        mask = np.random.rand(len(X)) < sigma # Get mask for sigma% of columns
        if mask.any():
            deltas = np.random.choice([-1, 1], size=mask.sum())
            cur = X.loc[X.index[mask], col].values
            deltas = np.where(cur == 8, -1, deltas) # If already 8, force delta to be -1 (as it cannot be higher than 8, from UCI)
            deltas = np.where(cur == -2, 1, deltas) # If already 2, force delta to be +1 (as it cannot be lower than -2, from UCI)
            X.loc[X.index[mask], col] = (cur + deltas).astype(X[col].dtype)

    return X

def full_preprocess_train(X_raw):
    """
    Full pipeline: winsorize (on own distribution) and fit ColumnTransformer.
    For the noisy model, the scaler is fitted on the noisy data itself.
    This mirrors what would happen in production if training data was corrupted.
    """
    X = X_raw.copy()

    # WINSORIZING

    for col in [f"PAID_TO_REMAINING_{m}" for m in ["SEP","AUG","JUL","JUN","MAY","APR"]]:
        X[col] = X[col].clip(upper=2.0)

    X["CREDIT_UTIL_MEAN"] = X["CREDIT_UTIL_MEAN"].clip(0.0, 1.5)
    X["CREDIT_UTIL_TREND"] = X["CREDIT_UTIL_TREND"].clip(-0.3, 0.3)

    for col in [f"PAY_AMT{m}" for m in ["SEP","AUG","JUL","JUN","MAY","APR"]]:
        X[col] = X[col].clip(upper=X[col].quantile(0.99))

    # PREPROCESSING
    pp = sklearn.base.clone(preprocessor)
    X_scaled = pp.fit_transform(X)
    
    return pd.DataFrame(X_scaled, columns=numeric_and_ordinal, index=X_raw.index)


# BASE MODELS: train once on clean data

y_train = train_target.values.ravel()
X_clean_df = full_preprocess_train(train_input_raw)

base_importances = {}
base_top_k_idx = {}
base_top_k_names = {}

for model_name, model in models.items():
    m = sklearn.base.clone(model)
    m.fit(X_clean_df, y_train)

    imp = get_importances(m)
    base_importances[model_name] = imp

    base_top_k_idx[model_name] = {}
    base_top_k_names[model_name] = {}

    # Use abs() for ranking of "importances" (LR)
    ranking = np.argsort(np.abs(imp))[::-1]

    for k in top_ks:
        idx = ranking[:k]
        base_top_k_idx[model_name][k] = idx
        base_top_k_names[model_name][k] = [numeric_and_ordinal[i] for i in idx]


# NOISED TRAINING LOOP

np.random.seed(42)

feat_imp_results = {}

for model_name, model in models.items():

    print(f"Training {model_name}")

    mean_cos = {k: [] for k in top_ks}
    std_cos = {k: [] for k in top_ks}


    for sigma in sigmas:
        iter_cos = {k: [] for k in top_ks}
        for _ in range(n_iterations):

            # Inject noise
            X_noisy_raw = inject_noise_train(train_input_raw, sigma)

            # Preprocess
            X_noisy_df = full_preprocess_train(X_noisy_raw)

            m = sklearn.base.clone(model)
            m.fit(X_noisy_df, y_train)

            v_base = base_importances[model_name]
            v_noisy = get_importances(m)

            # Calculate similarity for every K
            for k in top_ks:
                idx = base_top_k_idx[model_name][k]

                # Get weights of top K features before and after noise
                v_base_k = v_base[idx]
                v_noisy_k = v_noisy[idx]

                # Calculate cosine similarity between before and after
                cos = cosine_similarity(v_base_k, v_noisy_k)
                iter_cos[k].append(cos)

        # Aggregate for top Ks
        for k in top_ks:
            mean_cos[k].append(np.mean(iter_cos[k]))
            std_cos[k].append(np.std(iter_cos[k]))

            print(f"{model_name} | Sigma {sigma:.2f} | Top-{k} Cosine Similarity: {mean_cos[k][-1]:.4f} ± {2*std_cos[k][-1]:.4f}")

    # Aggregate for model
    feat_imp_results[model_name] = {
        "sigmas": np.array(sigmas),
        "mean_cos": {k: np.array(v) for k, v in mean_cos.items()},
        "std_cos": {k: np.array(v) for k, v in std_cos.items()}
    }


### PLOTS

lr_models  = ["LR_1", "LR_2"]
rf_models  = ["RF_1", "RF_2"]
xgb_models = ["XGB_1", "XGB_2"]

groups = [
    ("LR", lr_models),
    ("RF", rf_models),
    ("XGB", xgb_models)
]

# PLOT cosine similarity Top-5 vs Top-10

fig, axes = plt.subplots(3, 2, figsize=(21, 34))

colors = {5: "green", 10: "blue"}

for i, (name, model_pair) in enumerate(groups):
    for j in range(2): # 3x2 grid
        model_name = model_pair[j]
        res = feat_imp_results[model_name]

        for k in top_ks:
            mean = res["mean_cos"][k]
            std = res["std_cos"][k]

            axes[i,j].plot(
                res["sigmas"],
                mean,
                linewidth=2,
                label=f"Top-{k}",
                color=colors[k]
            )

            axes[i,j].fill_between(
                res["sigmas"],
                mean - 2*std,
                mean + 2*std,
                alpha=0.15,
                color=colors[k]
            )

        axes[i,j].set_title(f"{name} - {model_name}")
        axes[i,j].grid(True, linestyle=":")
        axes[i,j].set_ylim(0.6, 1.1)

        axes[i,j].xaxis.set_major_locator(ticker.MultipleLocator(0.05))
        axes[i,j].yaxis.set_major_locator(ticker.MultipleLocator(0.05))

        axes[i,j].set_xlabel("Sigma")
        axes[i,j].set_ylabel("Cosine Similarity")

        axes[i,j].legend()



plt.suptitle("Feature Importance Stability Under Training Noise", fontsize=28, y=0.96)
plt.tight_layout(rect=[0,0,1,0.96])
plt.savefig("./tests/similarity_under_noise/cosine_similarity.png")


# PLOT 2 (Top-10 features from each base model)

fig, axes = plt.subplots(3, 2, figsize=(21, 34))

plot_k = 10

for i, (name, model_pair) in enumerate(groups):
    for j in range(2):
        model_name = model_pair[j]

        tk = base_top_k_idx[model_name][plot_k]
        names = base_top_k_names[model_name][plot_k]
        vals = base_importances[model_name][tk]
        axes[i, j].barh(
            names[::-1],
            vals[::-1],
            color="blue"
        )

        axes[i, j].axvline(
            0,
            color="black",
            linewidth=0.8
        )

        axes[i, j].set_title(f"{name} - {model_name} (Top-{plot_k})")

        axes[i, j].set_xlabel("Coefficient (LR) / Importance (RF/XGB)")

        axes[i, j].grid(
            True,
            linestyle=":",
            axis="x"
        )


plt.suptitle("Top-10 Feature Importances (Clean Training Data)", fontsize=28, y=0.96)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("./tests/similarity_under_noise/base_models_top_features.png")

# Save results
joblib.dump(feat_imp_results, "./tests/similarity_under_noise/results.pkl")
```

---

Results:

![Cosine similarity graph (in /tests/similarity_under_noise/cosine_similarity.png)](./tests/similarity_under_noise/cosine_similarity.png)

![Base feature importances (in /tests/similarity_under_noise/base_models_top_features.png)](./tests/similarity_under_noise/base_models_top_features.png)

To properly interpret the stability curves, we must analyze how the underlying geometry of each architecture responds to progressive noise injection. The empirical data highlights different engineering behaviors across our model lineup.

### 1. The L1 Lasso Instability of Logistic Regression (The "Lasso Lottery")

Logistic Regression (`LR_1`, as it's our selected Noise-Resistant model) exhibits moderate structural volatility under noise, collapsing its feature importance vectors at around $0.15 \sigma$.
This behavior can be explained as a direct consequence of how class-weight proportions interact with $L1$ regularization when mildly correlated features are present:

> **The "Lasso Lottery" Mechanism:** 
> Our exploratory data analysis confirmed that while severe multicollinearity is absent, the historical tracking blocks (`PAY_*` and `PAY_AMT*` columns) exhibit moderate correlation ($0.5$ to $0.8$). Under heavy $L1$ (Lasso) regularization, the optimizer is forced to make sparse selections, tending to arbitrarily assign weight to a single "representative" variable in a correlated group while driving the rest to zero. On clean data, this creates a highly sparse baseline vector where only a few features "win" the lottery.

* **Why `LR_1` "Collapses" ($0.70 - 0.75$):** 
  With more and more added noise, the coordinate descent solver (LR) violently swaps its active $L1$ selection between different correlated variables across Monte Carlo iterations, as it is now almost impossible to determine the "real" best "all-in-one grouping" feature among the correlated ones. Because the resulting vectors are extremely sparse, this rapid swapping of coordinates causes the noisy weight vector to rotate almost completely away from the baseline, dragging the Cosine Similarity down.

### 2. The Diversification of Random Forest

Random Forest models (`RF_1` and `RF_2`) show a smooth, controlled degradation profile, with top-10 feature importance stability remaining high (≥0.97) even at extreme noise injection ($\sigma = 0.50$).

This is primarily a **variance-reduction effect from Bootstrap Aggregation**. Averaging feature importances across hundreds of such trees cancels out this per-tree noise the same way it cancels out prediction noise. The resulting ensemble-level importance ranking is therefore far more stable than any single tree's ranking would be (this is the standard bagging variance-reduction argument applied to importance vectors rather than to predictions).

### 3. The Error-Correcting Gradient Boosting Architecture

XGBoost models (`XGB_1` and `XGB_2`) also establish themselves with good structural stability under realistic noise levels, maintaining a flat stability curve throughout the majority of the stress test.

This performance is driven by its key algorithmic safeguard: because XGBoost learns sequentially by fitting residuals, the impact of random noise on any single feature is heavily dampened by the learning rate ($\eta$) and regularization penalties ($\gamma$, $\lambda$), and it's much less independently propagated as in Random Forest


All chosen models appear to be stable under moderate noise (i.e. $\sigma < 0.15/0.20$), thus we can confirm the robustness of these models under training noise and that they're not a random noisy artifact with good confidence, locking them as part of our final model line-up

## 5. SHAP Interpretability Assessment

To ensure that all of our selected models follow plausible default prediction strategies, we move beyond mere predictive accuracy to perform a rigorous **Interpretability Assessment**. Since a model can achieve high performance by exploiting artifacts, noise, or spurious correlations in the training data, SHAP analysis serves as our primary defense mechanism to validate that the learned logic aligns with established financial risk principles.

*   **Global Logic Validation:** We utilize **SHAP Summary Plots** to verify that the model’s global feature importance adheres to domain expertise. We systematically check for "Red Flags" in the feature contribution patterns, such as the model assigning significant predictive weight to variables that should be economically neutral, or exhibiting counter-intuitive relationships
*   **Plausibility Stress-Testing:** We test whether the model’s internal logic is **monotonically consistent** with traditional credit scoring theory. For instance, we confirm that across the cohort, an increase in "Payment Delay" consistently shifts the SHAP contribution toward higher default risk, while "Debt Coverage" improvements act as a risk mitigator. 
*   **Stability of Explanation:** We quantify the consistency of feature importance across different model versions. By comparing SHAP values of our candidate architectures (XGBoost, Random Forest, Logistic Regression), we ensure that the decision-making process is rooted in the underlying structure of the data rather than being an artifact of a specific optimization algorithm.

---

Code used to generate the graph:

```python
# Ignore parallelism sklearn warnings for SHAP
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn.utils.parallel")

### INIT
shap.initjs()
np.random.seed(42)


### PRIOR RECALIBRATION
# Saerens et al. 2002 prior shift. In log-odds space this is a constant additive
# offset: logit(p_cal) = logit(p_raw) + ln(w_pos / w_neg).
# We apply this offset to the SHAP base value before telescoping, so that the
# waterfall sums exactly to the calibrated probability while per-feature SHAP
# values in log-odds space remain unchanged.
PI_TRAIN = 0.2213
PI_REAL = 0.003

W_POS = PI_REAL / PI_TRAIN
W_NEG = (1 - PI_REAL) / (1 - PI_TRAIN)
LOG_ODDS_SHIFT = np.log(W_POS / W_NEG)

def calibrate_probabilities(p_raw, w_pos=W_POS, w_neg=W_NEG):
    p_raw = np.asarray(p_raw)
    num = p_raw * w_pos
    den = num + (1 - p_raw) * w_neg
    return num / den


### DATA PREPARATION
X_full_raw = train_input_raw.copy()
y_full = train_target.values.ravel()

coverage_cols = [f"PAID_TO_REMAINING_{m}" for m in ["SEP","AUG","JUL","JUN","MAY","APR"]]
for col in coverage_cols:
    X_full_raw[col] = X_full_raw[col].clip(upper=2.0)

X_full_raw["CREDIT_UTIL_MEAN"] = X_full_raw["CREDIT_UTIL_MEAN"].clip(0.0, 1.5)
X_full_raw["CREDIT_UTIL_TREND"] = X_full_raw["CREDIT_UTIL_TREND"].clip(-0.3, 0.3)

pay_amt_cols = [f"PAY_AMT{m}" for m in ["SEP","AUG","JUL","JUN","MAY","APR"]]
for col in pay_amt_cols:
    q99 = X_full_raw[col].quantile(0.99)
    X_full_raw[col] = X_full_raw[col].clip(upper=q99)


### PREPROCESSING
features = list(X_full_raw.columns)

preprocessor = sklearn.compose.ColumnTransformer(
    transformers=[("num", sklearn.preprocessing.RobustScaler(), features)],
    remainder="drop"
)

X_processed = pd.DataFrame(
    preprocessor.fit_transform(X_full_raw),
    columns=features,
    index=X_full_raw.index
)


### SHAP SAMPLE + BACKGROUND
MAX_SHAP_SAMPLES = 1000
BACKGROUND_SAMPLES = 1000

sss = sklearn.model_selection.StratifiedShuffleSplit(
    n_splits=1, train_size=MAX_SHAP_SAMPLES, random_state=42
)
for shap_idx, _ in sss.split(X_processed, y_full):
    X_shap_sample = X_processed.iloc[shap_idx]

bg_sss = sklearn.model_selection.StratifiedShuffleSplit(
    n_splits=1, train_size=BACKGROUND_SAMPLES, random_state=42
)
for bg_idx, _ in bg_sss.split(X_processed, y_full):
    X_background = X_processed.iloc[bg_idx]

# KernelExplainer scales poorly with large background; 200 samples are sufficient.
X_background_kernel = X_background.sample(n=200, random_state=42)


### HELPER: log-odds -> probability via telescoping sum
# Guarantees exact additivity in probability space by construction.
# Individual probability contributions are order-dependent; we fix order by
# descending |log-odds contribution| per sample. Shapley axioms hold in log-odds
# space, not in the converted probability space.
def logodds_shap_to_probability(shap_values_logodds, base_values_logodds):
    shap_values_logodds = np.asarray(shap_values_logodds)
    n_samples, n_features = shap_values_logodds.shape

    base_values_logodds = np.broadcast_to(
        np.asarray(base_values_logodds), (n_samples,)
    ).astype(float)
    base_prob = expit(base_values_logodds)

    prob_values = np.zeros_like(shap_values_logodds)

    for i in range(n_samples):
        row = shap_values_logodds[i]
        order = np.argsort(-np.abs(row), kind="stable")

        cum_logit = base_values_logodds[i]
        cum_prob = base_prob[i]

        for feat_idx in order:
            cum_logit += row[feat_idx]
            new_prob = expit(cum_logit)
            prob_values[i, feat_idx] = new_prob - cum_prob
            cum_prob = new_prob

    return prob_values, base_prob


### SHAP CALCULATION
all_shap_outputs = {}
explainer_outputs = {}

# --- Random Forest: KernelExplainer on logit(proba) for exact log-odds SHAP ---
print("Running SHAP: RF")

def rf_logit(x):
    # KernelExplainer passes numpy arrays; wrap to DataFrame to preserve feature names.
    if not isinstance(x, pd.DataFrame):
        x = pd.DataFrame(x, columns=features)
    p = rf_model_1.predict_proba(x)[:, 1]
    p = np.clip(p, 1e-15, 1 - 1e-15)
    return logit(p)

rf_kernel_explainer = shap.KernelExplainer(rf_logit, X_background_kernel.values)
rf_shap_logodds = rf_kernel_explainer.shap_values(
    X_shap_sample.values, nsamples=4096
)

# Apply calibration shift to base value only; feature contributions unchanged.
rf_base_logodds_cal = rf_kernel_explainer.expected_value + LOG_ODDS_SHIFT
rf_prob_values, rf_base_prob = logodds_shap_to_probability(
    rf_shap_logodds, rf_base_logodds_cal
)

rf_shap_prob = shap.Explanation(
    values=rf_prob_values,
    base_values=rf_base_prob,
    data=X_shap_sample.values,
    feature_names=features
)

all_shap_outputs["RF"] = {
    "shap_values": rf_shap_prob,
    "shap_values_logodds": rf_shap_logodds,
    "base_value_logodds_raw": rf_kernel_explainer.expected_value,
    "base_value_logodds_cal": rf_base_logodds_cal,
}
explainer_outputs["RF"] = {"explainer": rf_kernel_explainer}

# --- Logistic Regression: LinearExplainer in log-odds, then telescoping ---
print("Running SHAP: LR")

lr_explainer = shap.LinearExplainer(
    lr_model_1,
    X_background,
    feature_perturbation="interventional"
)
lr_shap_logodds = lr_explainer(X_shap_sample)

lr_base_logodds_cal = lr_shap_logodds.base_values + LOG_ODDS_SHIFT
lr_prob_values, lr_base_prob = logodds_shap_to_probability(
    lr_shap_logodds.values, lr_base_logodds_cal
)

lr_shap_prob = shap.Explanation(
    values=lr_prob_values,
    base_values=lr_base_prob,
    data=X_shap_sample.values,
    feature_names=features
)

all_shap_outputs["LR"] = {
    "shap_values": lr_shap_prob,
    "shap_values_logodds": lr_shap_logodds,
    "base_value_logodds_cal": lr_base_logodds_cal,
}
explainer_outputs["LR"] = {"explainer": lr_explainer}


### SAVE
joblib.dump(all_shap_outputs, "./tests/shap_analysis/results.pkl")
joblib.dump(explainer_outputs, "./tests/shap_analysis/explainers.pkl")

X_shap_sample.to_csv("./data/tests/shap_sample.csv", index=True)
X_background.to_csv("./data/tests/shap_background.csv", index=True)


### CHECK ADDITIVITY (against calibrated probabilities)
rf_reconstructed = rf_shap_prob.base_values + rf_shap_prob.values.sum(axis=1)
rf_actual_cal = calibrate_probabilities(rf_model_1.predict_proba(X_shap_sample)[:, 1])
print("RF additivity max abs error (calibrated):", np.max(np.abs(rf_reconstructed - rf_actual_cal)))

lr_reconstructed = lr_shap_prob.base_values + lr_shap_prob.values.sum(axis=1)
lr_actual_cal = calibrate_probabilities(lr_model_1.predict_proba(X_shap_sample)[:, 1])
print("LR additivity max abs error (calibrated):", np.max(np.abs(lr_reconstructed - lr_actual_cal)))


### SUMMARY PLOTS
for model_name in ["RF", "LR"]:
    shap.summary_plot(
        all_shap_outputs[model_name]["shap_values"],
        X_shap_sample,
        max_display=21,
        plot_type="dot",
        show=False
    )
    plt.title(model_name, fontsize=18)
    plt.subplots_adjust(left=0.25, right=0.95, top=0.95, bottom=0.05)
    plt.savefig(f"./tests/shap_analysis/{model_name}_SHAP_summary.png", bbox_inches="tight")
    plt.close()


### PER-CUSTOMER CALIBRATED PROBABILITY (audit metadata)
calibrated_probs = pd.DataFrame({
    "RF_p_raw": rf_model_1.predict_proba(X_shap_sample)[:, 1],
    "RF_p_calibrated": calibrate_probabilities(
        rf_model_1.predict_proba(X_shap_sample)[:, 1]
    ),
    "LR_p_raw": lr_model_1.predict_proba(X_shap_sample)[:, 1],
    "LR_p_calibrated": calibrate_probabilities(
        lr_model_1.predict_proba(X_shap_sample)[:, 1]
    ),
}, index=X_shap_sample.index)

calibrated_probs.to_csv("./data/tests/shap_calibrated_probs.csv", index=True)
```

---

Results:

![SHAP Beeswarm for LR (in /tests/shap_analysis/LR_SHAP_summary.png)](./tests/shap_analysis/LR_SHAP_summary.png)

![SHAP Beeswarm for RF (in /tests/shap_analysis/RF_SHAP_summary.png)](./tests/shap_analysis/RF_SHAP_summary.png)

RF additivity max abs error (calibrated): 2.498001805406602e-16

LR additivity max abs error (calibrated): 2.42861286636753e-17

### 1. Qualitative Analysis of Model Behaviors

We perform a qualitative analysis of the generated SHAP beeswarm plots to validate that the learned decision strategies align with established financial risk principles and to identify the model-specific "logic" of each candidate.

- **Logistic Regression (`LR_1`):**
    - Operates as a transparent, additive model where every feature exerts a fixed influence on the log-odds of default, independent of the applicant's overall profile.
    - **Risk Drivers:** `PAY_SEP`, `PAY_AUG` and `PAY_JUL` exhibit positive coefficients; higher month-delay values strictly increase the probability of default.
    - **Mitigation Drivers:** `PAY_AMT` variables across all months show a consistent negative impact; lower payment volumes systematically elevate the default risk score.
    - *Strategic Value:* Provides a highly auditable baseline, ensuring compliance with simple, monotonic financial logic.

- **Random Forest (`RF_1`):**
    - Exhibits higher feature sparsity compared to the linear baseline, activating a broader range of variables to refine risk stratification.
    - **Temporal Delays (`PAY_*`):** Mirrors the linear model’s logic where extreme delays (high values) positively correlate with default risk. However, for `PAY_SEP`, median/average values actually tend to lower more the risk of default than the few perfect / no-utilization customers (more of that later*)
    - **Payment Dynamics (`PAY_AMT*`):** Demonstrates non-linear behavior; while lower payments generally drive positive or mildly negative risk contributions, the model isolates high-value observations as strong negative-risk indicators, suggesting a threshold effect for high-liquidity clients.
    - **Payment Coverage (`PAID_TO_REMAINING_*`):** Ranked among the least important features. The obsearved bell-shaped distribution of SHAP values (similar to the one of `PAY_AMT*` features) suggests these variables lack predictive power in isolation, likely due to a high density of customers clustering around moderate coverage levels. The few high-value outliers consistently fall on the left side of the plot (i.e. decreasing risk observations).
    - **Credit Balance (`LIMIT_BAL`):** Displays a moderately inverse relationship: lower balances are generally associated with higher default risk (likely representing newer or high-risk segments), while median/average credit limit customers consistently fall close to the SHAP zero-point.
    - **Utilization Metrics (`CREDIT_UTIL_MEAN` & `CREDIT_UTIL_TREND`):**
        - `CREDIT_UTIL_MEAN` reveals, apart from a few outliers, a logical explaination: higher credit utilization correlates to a higher risk of default
        - `CREDIT_UTIL_TREND` displays the majority of mass clustered just below SHAP zero-point, suggesting that, for most of the population, trend volatility is not a primary driver of default. Contrary to common belief, the few high-value outliers don't directly correlate with higher risks, while the more risk-driving observations are found with the lowest trends (i.e. decreasing balances)

### 2. The Masking Hypothesis *

The observed ambiguities in `PAY_*` features and in the lowest values for `CREDIT_UTIL_TREND` in Random Forest is likely not statistical noise, but the potential identification of a particular credit-default pattern. We hypothesize that these models have detected a distinct cohort of **strategic defaulters** that demonstrate seemingly impeccable financial health and over-paying current installments as a "masking" behavior, likely preparatory to a sudden, high-magnitude credit utilization followed by an immediate default.

*Note on Validation: While this observation remains a working hypothesis subject to further longitudinal validation, it aligns with known "bust-out fraud" attack vectors documented in credit risk literature. The emergence of these patterns in non-linear ensemble models, which remain invisible to linear baselines, underscores the critical role of high-dimensional XAI algorithms (such as SHAP) in identifying non-obvious, potentially malicious credit behaviors.*

### 3. Final decision

Generally speaking though, all selected models seem to follow logical patterns in credit usage to identify defaulters. There are no features that exhibit radically different logic from what real banking systems used and currently used, so we confirm once again our model choice

## 6. Production deployment

After having validated our models for production use and having separeted different architectures for different scenarios (Primary, Robust, Fail-safe), we proceed with deployment.

This will consist of data, pipeline and model saving, along with a working demo on Hugging Face spaces (which will be found at [this](https://huggingface.co/spaces/freyflyy/taiwan-robust-explainable-credit-lend) link) which will enable anyone to try the final model lineup to predict defaulting customers, with the option to generate an automatic PDF report explaining the model's decision with waterfall plots and per-feature analysis

### 6.1 Saving and Final evaluation

As a final test, the performance of our Primary model is calculated on the last test dataset, which remained untouched to avoid any overfitting on the final performance

In [37]:
### SAVING

os.makedirs("./models/final", exist_ok=True)

numeric_and_ordinal = [col for col in train_input_raw.columns]

train_input_final = train_input_raw.copy()

# 1. PAID_TO_REMAINING_* -> cap at 2.0
coverage_cols = [f"PAID_TO_REMAINING_{m}" for m in ["SEP","AUG","JUL","JUN","MAY","APR"]]
for col in coverage_cols:
    train_input_final[col] = train_input_final[col].clip(upper=2.0)

# 2. CREDIT_UTIL_MEAN -> floor/cap [0, 1.5]
train_input_final["CREDIT_UTIL_MEAN"] = train_input_final["CREDIT_UTIL_MEAN"].clip(lower=0.0, upper=1.5)

# 3. CREDIT_UTIL_TREND -> cap [-0.3, 0.3]
train_input_final["CREDIT_UTIL_TREND"] = train_input_final["CREDIT_UTIL_TREND"].clip(lower=-0.3, upper=0.3)

# 4. PAY_AMT* -> cap at 99th percentile (train)
pay_amt_cols = [f"PAY_AMT{m}" for m in ["SEP","AUG","JUL","JUN","MAY","APR"]]
thresholds_q99 = {}
for col in pay_amt_cols:
    q99 = train_input_final[col].quantile(0.99)
    thresholds_q99[col] = q99
    train_input_final[col] = train_input_final[col].clip(upper=q99)

# Preprocessor fitted on the training set (to be used for test set)
preprocessor = sklearn.compose.ColumnTransformer(
    transformers=[
        ('num', sklearn.preprocessing.RobustScaler(), numeric_and_ordinal),
    ],
    remainder='drop'
)
preprocessor.fit(train_input_final)
joblib.dump(preprocessor, "./models/final/preprocessor.joblib")

# Save the thresholds for winsorization
joblib.dump(thresholds_q99, "./models/final/pay_amt_99th.joblib")

# Decision thresholds
decision_thresholds = {
    "LR": 0.006,
    "RF": 0.017
}
joblib.dump(decision_thresholds, "./models/final/decision_thresholds.joblib")

# Models
LR = joblib.load("./models/LR_Model_1.joblib")
RF = joblib.load("./models/RF_Model_1.joblib")

joblib.dump(LR, "./models/final/LR.joblib")
joblib.dump(RF, "./models/final/RF.joblib")

['./models/final/RF.joblib']

---

Code:

```python
### DATA AND MODEL LOADING

os.makedirs("./tests", exist_ok=True)

test_input_raw = pd.read_csv("./data/splits/raw/test_input_raw.csv", index_col=0)
test_target    = pd.read_csv("./data/splits/preprocessed/test_target.csv", index_col=0)
test_EAD       = pd.read_csv("./data/splits/EADs/test_EAD.csv", index_col=0)

rf_model        = joblib.load('./models/final/RF.joblib')
lr_model        = joblib.load('./models/final/LR.joblib')
preprocessor    = joblib.load('./models/final/preprocessor.joblib')
thresholds_q99  = joblib.load('./models/final/pay_amt_99th.joblib')
decision_thresh = joblib.load('./models/final/decision_thresholds.joblib')
thresh_rf       = decision_thresh["RF"]
thresh_lr       = decision_thresh["LR"]


### FUNCTIONS

def calibrate_probabilities(p_raw, pi_train=0.2213, pi_real=0.003):
    p_raw = np.array(p_raw).flatten()
    w_pos = pi_real / pi_train
    w_neg = (1 - pi_real) / (1 - pi_train)
    num = p_raw * w_pos
    den = num + (1 - p_raw) * w_neg
    return num / den

def custom_value_metric(y_true, y_pred, ead_vector, pi_train=0.2213, pi_real=0.003):
    y_true = np.array(y_true).flatten()
    y_pred = np.array(y_pred).flatten()
    ead_vector = np.array(ead_vector).flatten()
    w_pos = pi_real / pi_train
    w_neg = (1 - pi_real) / (1 - pi_train)
    LGD, mitigation_cost_factor, gain_factor = 0.70, 0.10, 0.01
    tn_mask = (y_true == 0) & (y_pred == 0)
    fn_mask = (y_true == 1) & (y_pred == 0)
    tp_mask = (y_true == 1) & (y_pred == 1)
    tn_revenue = np.sum(np.maximum(ead_vector[tn_mask], 0) * gain_factor) * w_neg
    fn_loss    = -np.sum(np.maximum(ead_vector[fn_mask], 0) * LGD) * w_pos
    tp_cost    = -np.sum(np.maximum(ead_vector[tp_mask], 0) * mitigation_cost_factor) * w_pos
    return tn_revenue + fn_loss + tp_cost


### PIPELINE PREPARATION

raw_continuous_cols = [
    'LIMIT_BAL',
    'PAY_AMTSEP', 'PAY_AMTAUG', 'PAY_AMTJUL', 'PAY_AMTJUN', 'PAY_AMTMAY', 'PAY_AMTAPR',
    'PAID_TO_REMAINING_SEP', 'PAID_TO_REMAINING_AUG', 'PAID_TO_REMAINING_JUL',
    'PAID_TO_REMAINING_JUN', 'PAID_TO_REMAINING_MAY', 'PAID_TO_REMAINING_APR',
    'CREDIT_UTIL_MEAN', 'CREDIT_UTIL_TREND'
]
raw_discrete_cols = ['PAY_SEP', 'PAY_AUG', 'PAY_JUL', 'PAY_JUN', 'PAY_MAY', 'PAY_APR']
numeric_and_ordinal = [col for col in test_input_raw.columns]

def winsorize(X_raw):
    X = X_raw.copy()
    for col in [f"PAID_TO_REMAINING_{m}" for m in ["SEP", "AUG", "JUL", "JUN", "MAY", "APR"]]:
        X[col] = X[col].clip(upper=2.0)
    X["CREDIT_UTIL_MEAN"]  = X["CREDIT_UTIL_MEAN"].clip(0.0, 1.5)
    X["CREDIT_UTIL_TREND"] = X["CREDIT_UTIL_TREND"].clip(-0.3, 0.3)
    for col, q99_val in thresholds_q99.items():
        X[col] = X[col].clip(upper=q99_val)
    return X

# Preprocess clean test set once for RF
X_test_clean = winsorize(test_input_raw)
X_test_clean_df = pd.DataFrame(
    preprocessor.transform(X_test_clean),
    columns=numeric_and_ordinal, index=X_test_clean.index
)

# RF predictions (deterministic, computed once)
proba_rf_full = calibrate_probabilities(rf_model.predict_proba(X_test_clean_df)[:, 1])
pred_rf_full  = (proba_rf_full >= thresh_rf).astype(int)
y_true_full   = test_target.values.ravel()
ead_full      = test_EAD.values.ravel()

# Noise scales computed on the test set
noise_scales = {}
for col in raw_continuous_cols:
    l, u = np.percentile(test_input_raw[col], [1.5, 98.5])
    noise_scales[col] = u - l

def inject_noise(X_raw, sigma):
    X = X_raw.copy()
    if sigma == 0:
        return X
    n = len(X)
    for col in raw_continuous_cols:
        X[col] += np.random.normal(0, sigma * noise_scales[col], size=n)
    for col in raw_discrete_cols:
        mask = np.random.rand(n) < sigma
        if mask.any():
            deltas = np.random.choice([-1, 1], size=mask.sum())
            pos = np.where(mask)[0]
            cur = X.iloc[pos][col].values
            deltas = np.where(cur == 8, -1, deltas)
            deltas = np.where(cur == -2, 1, deltas)
            X.iloc[pos, X.columns.get_loc(col)] = (cur + deltas).astype(X[col].dtype)
    return X


### DUAL BOOTSTRAP

np.random.seed(42)
n_boot = 10_000
rng    = np.random.default_rng(42)
n_test = len(test_input_raw)

sigmas  = np.round(np.arange(0, 0.51, 0.05), 2)
weights = np.array([0.10, 0.15, 0.20, 0.15, 0.10, 0.08, 0.07, 0.05, 0.05, 0.03, 0.02])

boot_rf = np.empty(n_boot)
boot_lr = np.empty(n_boot)

for b in range(n_boot):
    idx = rng.integers(0, n_test, n_test)

    # --- RF_1 NORMAL (clean, deterministic) ---
    boot_rf[b] = custom_value_metric(
        y_true_full[idx], pred_rf_full[idx], ead_full[idx]
    ) / n_test

    # --- LR_1 NOISY (stochastic noise, redrawn every iteration) ---
    X_b   = test_input_raw.iloc[idx]
    y_b   = test_target.iloc[idx].values.ravel()
    ead_b = test_EAD.iloc[idx].values.ravel()
    n_b   = len(y_b)

    profits_sigma = []
    for sigma in sigmas:
        X_n = winsorize(inject_noise(X_b, sigma))
        X_n_df = pd.DataFrame(
            preprocessor.transform(X_n),
            columns=numeric_and_ordinal, index=X_n.index
        )
        p = calibrate_probabilities(lr_model.predict_proba(X_n_df)[:, 1])
        pred = (p >= thresh_lr).astype(int)
        profits_sigma.append(
            custom_value_metric(y_b, pred, ead_b) / n_b
        )
    boot_lr[b] = np.dot(profits_sigma, weights)

    if (b + 1) % 1000 == 0:
        print(f"Bootstrap progress: {b + 1:,}/{n_boot:,}")


### TEXT REPORT

BASELINE = 12.65
MONTHLY_CLIENTS = 2_000_000
SEP_LEN = 60

def fmt_money(val: float) -> str:
    return f"{val:,.0f}"

def report(name: str, arr: np.ndarray):
    mean, std = arr.mean(), arr.std()
    lo, hi = np.percentile(arr, [2.5, 97.5])
    uplift = mean - BASELINE
    uplift_lo, uplift_hi = lo - BASELINE, hi - BASELINE
    prob_above_baseline = np.mean(arr > BASELINE) * 100

    print(f"\n{'-' * SEP_LEN}")
    print(f"SCENARIO: {name}")
    print(f"{'-' * SEP_LEN}")
    print(f"  Profit/customer      : ${mean:6.2f}  (std: ${std:.2f})")
    print(f"  95% CI               : [${lo:.2f}, ${hi:.2f}]")
    print(f"  Uplift vs baseline    : ${uplift:6.2f}  (95% CI: [${uplift_lo:.2f}, ${uplift_hi:.2f}])")
    print(f"  P(profit > baseline) : {prob_above_baseline:5.1f}%")

    for label, months in [("Monthly", 1), ("Annual", 12)]:
        gain    = uplift * MONTHLY_CLIENTS * months
        gain_lo = uplift_lo * MONTHLY_CLIENTS * months
        gain_hi = uplift_hi * MONTHLY_CLIENTS * months
        print(f"  Net {label:<7s} gain     : +${fmt_money(gain)}  "
              f"(95% CI: [${fmt_money(gain_lo)}, ${fmt_money(gain_hi)}])")

    return mean, lo, hi, uplift, uplift_lo, uplift_hi, prob_above_baseline

res_rf = report("Normal - RF_1 (clean test set)", boot_rf)
res_lr = report("Noisy - LR_1 (uncertain world)", boot_lr)

# Paired comparison
delta = boot_rf - boot_lr
print(f"\n{'-' * SEP_LEN}")
print("PAIRED COMPARISON: RF Normal vs. LR Noisy")
print(f"{'-' * SEP_LEN}")
print(f"  Mean delta   : ${delta.mean():.2f}/customer")
print(f"  95% CI delta : [${np.percentile(delta, 2.5):.2f}, ${np.percentile(delta, 97.5):.2f}]")
print(f"  P(RF > LR)   : {np.mean(delta > 0) * 100:.1f}%")


### COMPARATIVE PLOT (FINAL)

plt.figure(figsize=(11, 5.5), dpi=120)

# Uplift data
rf_uplift = boot_rf - BASELINE
lr_uplift = boot_lr - BASELINE

# Dynamic range: zoom on the central mass (99.5% of the data)
all_uplift = np.concatenate([rf_uplift, lr_uplift])
x_lo = np.percentile(all_uplift, 0.25) - 0.02
x_hi = np.percentile(all_uplift, 99.75) + 0.02
bins = np.linspace(x_lo, x_hi, 100)

# Histograms with visible edges to emphasize individual bins
plt.hist(rf_uplift, bins=bins, alpha=0.50, color="#2ca02c",
         edgecolor="darkgreen", linewidth=0.7,
         label=f"RF Normal (μ=${res_rf[3]:.2f}, σ=${rf_uplift.std():.2f})")
plt.hist(lr_uplift, bins=bins, alpha=0.50, color="#ff7f0e",
         edgecolor="darkorange", linewidth=0.7,
         label=f"LR Noisy (μ=${res_lr[3]:.2f}, σ=${lr_uplift.std():.2f})")

# Reference lines
plt.axvline(0, color="black", linestyle="--", linewidth=2, label=f"Baseline (${BASELINE})")
plt.axvline(res_rf[3], color="#1a5c1a", linestyle="-", linewidth=2.5, zorder=5)
plt.axvline(res_lr[3], color="#cc6600", linestyle="-", linewidth=2.5, zorder=5)

plt.xlim(x_lo, x_hi)
plt.title("Bootstrap Uplift Comparison", fontsize=12)
plt.xlabel("Profit uplift per customer per month ($)")
plt.ylabel("Frequency (bootstrap resamples)")
plt.legend(loc="upper right", framealpha=0.95)
plt.grid(axis="y", alpha=0.25, linestyle=":")
plt.tight_layout()

plt.savefig("./tests/final_evaluation_bootstrap.png", bbox_inches="tight")
```

---

Results:

![Final evaluation graph (in /tests/final_evaluation_bootstrap.png)](./tests/final_evaluation_bootstrap.png)

```text
------------------------------------------------------------
SCENARIO: Normal - RF_1 (clean test set)
------------------------------------------------------------
  Profit/customer      : $ 12.93  (std: $0.40)
  95% CI               : [$12.16, $13.72]
  Uplift vs baseline    : $  0.28  (95% CI: [$-0.49, $1.07])
  P(profit > baseline) :  76.0%
  Net Monthly gain     : +$567,366  (95% CI: [$-984,416, $2,148,323])
  Net Annual  gain     : +$6,808,393  (95% CI: [$-11,812,997, $25,779,877])

------------------------------------------------------------
SCENARIO: Noisy - LR_1 (uncertain world)
------------------------------------------------------------
  Profit/customer      : $ 12.74  (std: $0.40)
  95% CI               : [$11.97, $13.52]
  Uplift vs baseline    : $  0.09  (95% CI: [$-0.68, $0.87])
  P(profit > baseline) :  59.1%
  Net Monthly gain     : +$180,761  (95% CI: [$-1,365,709, $1,740,127])
  Net Annual  gain     : +$2,169,134  (95% CI: [$-16,388,503, $20,881,526])

------------------------------------------------------------
PAIRED COMPARISON: RF Normal vs. LR Noisy
------------------------------------------------------------
  Mean delta   : $0.19/customer
  95% CI delta : [$0.06, $0.33]
  P(RF > LR)   : 99.8%
```

The bootstrap results support a cautious but positive conclusion.

- **Absolute uplift is directionally positive but not tightly bounded:** Both scenarios
show mean profit above the $12.65 baseline (RF Normal: +$0.28/cust, LR Noisy:
+$0.09/cust), but neither 95% CI excludes zero (RF: [-$0.49, $1.07], LR:
[-$0.68, $0.87]). P(>baseline) is 76.0% for RF and only 59.1% for LR, meaningfully
better than a coin flip, but not evidence of a guaranteed improvement at the customer
level. This is expected: EAD is heavy-tailed and the calibrated event rate (0.3%) is
low, so per-customer profit variance dominates the mean at n≈6'000.

- **The paired comparison is the strongest result in this section:** Because both
scenarios are evaluated on the same bootstrap resample at each iteration, the paired
delta (RF Normal − LR Noisy) removes shared sampling noise: mean +$0.19/cust,
95% CI [$0.06, $0.33], P(RF > LR) = 99.8%. This is the one claim in this analysis
that holds up at conventional significance. Primary (RF_1) reliably outperforms
Robust (LR_1) when RF operates under clean conditions and LR under adverse ones,
which is exactly the operating assumption behind the dual-engine architecture.

### 6.2 Practical interpretation

Scaled to an assumed 2M-customer monthly portfolio: the
Primary model maps to +$567k/month (95% CI [-$984k, +$2.15M]), +$6.81M/year (CI
[-$11.81M, +$25.78M]). The Robust model, evaluated under the noise-weighted
"Uncertain World" scenario, maps to +$181k/month (CI [-$1.37M, +$1.74M]), +$2.17M/year
(CI [-$16.39M, +$20.88M]), smaller and less certain, consistent with its role as the
degraded-conditions fallback rather than the primary revenue driver. Both point
estimates are economically meaningful, but the intervals are wide enough that this
should be read as "plausibly a meaningful gain, with real downside risk in a bad
draw" rather than a guaranteed return. Given the 0.3% real-world default prior is
itself an estimate (not fit from data), this uncertainty compounds an assumption
that is already a first-order approximation.

### 6.3 Recommendation

These figures justify a staged or partial rollout (e.g. shadow-mode or canary deployment against a subset of the live portfolio) rather than a full-scale switch, both to validate the calibration assumption against real outcomes and to narrow the confidence interval with production data before the uplift is reported as a committed number.

## 7. Final considerations

Through this project we moved from abstract predictive validation to an auditable, end-to-end decision pipeline. By combining leakage-free preprocessing, probability calibration against a realistic production prior, model-agnostic local explainability, and automated audit reporting, this single-author project connects raw predictive power to real-world regulatory defensibility.

Summary:

- A deterministic, leakage-free preprocessing pipeline, verified for cross-environment stability
- Three model classes trained with full grid search, two candidates retained per class, stress-tested for noise robustness, feature-dropping resistance, and feature-importance stability
- A model-agnostic local explainability layer (SHAP) translating black-box decisions into auditable, domain-consistent insights
- Automated structured audit report generation

**The actual economic result:** against the "accept everyone" baseline (\$12.65/cust/month), the production model yields an uplift of approximately **\$0.28/cust/month**. Scaled across an assumed base of 2 million monthly customers, this corresponds to almost **$7M/year in incremental profit**, a figure that is far from negligible in absolute terms, even though the relative lift over the naive baseline remains moderate (+2.2%).

This should be read for what it is: not a wholesale reinvention of credit risk scoring, but a measurable, reproducible improvement over a decision process that, in most real-world lending operations, still relies largely on static heuristics.

Every experiment in this repository, from all of the grid searches, Monte Carlo noise injections (50 iterations × 11 σ-levels × 5 folds × 6 models) and bootstraps, ran on a 2018 consumer laptop (i7-8550U, no GPU acceleration) without cloud compute. This is reflected directly in design choices: checkpointed/resumable grid search, exclusion of deep learning architectures, and n_boot/n_iterations sized to complete in single-session runtime.

---

The working demo can be found at [this](https://huggingface.co/spaces/freyflyy/taiwan-robust-explainable-credit-lend) link, while the full [GitHub repo](https://github.com/FreyFlyy/taiwan-robust-explainable-credit-lend-(ENG)) can be found in my [GitHub](https://github.com/FreyFlyy) profile
